In [ ]:
# Célula 1 — Configuração (rode primeiro)
import basedosdados as bd
import pandas as pd

BILLING_ID = "microdados-caged-510923"
ANO_INI, ANO_FIM = 2004, 2014

# Para testar rápido e barato, coloque uma UF (ex.: "AL"). Use None para o Brasil todo.
UF_TESTE = None


def filtro_uf(alias="d"):
    return f"AND {alias}.sigla_uf = '{UF_TESTE}'" if UF_TESTE else ""


def rodar(nome, query):
    """Roda a query e não derruba o notebook se der erro."""
    print(f">>> Baixando {nome}...")
    try:
        df = bd.read_sql(query=query, billing_project_id=BILLING_ID)
        print(f"    OK: {len(df):,} linhas | anos: {sorted(df['ano'].unique())}")
        return df
    except Exception as e:
        print(f"    ERRO em {nome}: {e}")
        return None


def colunas(tabela):
    """Lista as colunas de uma tabela da BD (útil se der erro de nome de coluna)."""
    dataset, tab = tabela.split(".")
    q = f"""
    SELECT column_name, data_type
    FROM `basedosdados.{dataset}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{tab}'
    """
    return bd.read_sql(query=q, billing_project_id=BILLING_ID)

In [ ]:
# Célula 2 — RAIS: emprego em 31/12 e salário médio por município-ano
query_rais = f"""
WITH base AS (
    SELECT
        d.ano,
        d.sigla_uf,
        d.id_municipio,
        COUNTIF(SAFE_CAST(d.vinculo_ativo_3112 AS INT64) = 1)            AS emprego_3112,
        AVG(IF(SAFE_CAST(d.vinculo_ativo_3112 AS INT64) = 1
               AND SAFE_CAST(d.valor_remuneracao_dezembro AS FLOAT64) > 0,
               SAFE_CAST(d.valor_remuneracao_dezembro AS FLOAT64), NULL)) AS salario_nominal_dez,
        AVG(IF(SAFE_CAST(d.valor_remuneracao_media AS FLOAT64) > 0,
               SAFE_CAST(d.valor_remuneracao_media AS FLOAT64), NULL))    AS salario_nominal
    FROM `basedosdados.br_me_rais.microdados_vinculos` AS d
    WHERE d.ano BETWEEN {ANO_INI} AND {ANO_FIM}
      {filtro_uf()}
    GROUP BY 1, 2, 3
)
SELECT
    b.ano,
    b.sigla_uf,
    b.id_municipio,
    m.nome AS nome_municipio,
    b.emprego_3112,
    b.salario_nominal,          -- remuneração média mensal no ano, em R$ correntes
    b.salario_nominal_dez       -- remuneração de dezembro dos vínculos ativos, em R$ correntes
FROM base AS b
LEFT JOIN (SELECT DISTINCT id_municipio, nome
           FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS m
    ON b.id_municipio = m.id_municipio
ORDER BY b.id_municipio, b.ano
"""

rais = rodar("RAIS", query_rais)

if rais is not None:
    rais.to_excel("RAIS_2004_2014.xlsx", index=False)   # precisa do openpyxl
    print("Salvo em RAIS_2004_2014.xlsx")
    display(rais.head())
else:
    print(colunas("br_me_rais.microdados_vinculos"))

In [ ]:
# Célula 3 — CAGED: admissões, desligamentos, saldo e salário de admissão por município-ano
query_caged = f"""
WITH base AS (
    SELECT
        d.ano,
        d.sigla_uf,
        d.id_municipio,
        COUNTIF(SAFE_CAST(d.saldo_movimentacao AS INT64) = 1)   AS admissoes,
        COUNTIF(SAFE_CAST(d.saldo_movimentacao AS INT64) = -1)  AS desligamentos,
        SUM(SAFE_CAST(d.saldo_movimentacao AS INT64))           AS saldo,
        AVG(IF(SAFE_CAST(d.saldo_movimentacao AS INT64) = 1
               AND SAFE_CAST(d.salario_mensal AS FLOAT64) > 0,
               SAFE_CAST(d.salario_mensal AS FLOAT64), NULL))   AS salario_nominal
    FROM `basedosdados.br_me_caged.microdados_antigos` AS d
    WHERE d.ano BETWEEN {ANO_INI} AND {ANO_FIM}
      {filtro_uf()}
    GROUP BY 1, 2, 3
)
SELECT
    b.ano,
    b.sigla_uf,
    b.id_municipio,
    m.nome AS nome_municipio,
    b.admissoes,
    b.desligamentos,
    b.saldo,
    b.salario_nominal           -- salário médio de admissão, em R$ correntes
FROM base AS b
LEFT JOIN (SELECT DISTINCT id_municipio, nome
           FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS m
    ON b.id_municipio = m.id_municipio
ORDER BY b.id_municipio, b.ano
"""

caged = rodar("CAGED", query_caged)

if caged is not None:
    caged.to_excel("CAGED_2004_2014.xlsx", index=False)  # precisa do openpyxl
    print("Salvo em CAGED_2004_2014.xlsx")
    display(caged.head())
else:
    print(colunas("br_me_caged.microdados_antigos"))